# Skin Cancer Detection Using Machine Learning & OpenCV Computer Vision
## Comprehensive Dermatological Feature Extraction, SVM Classification, and Test Evaluation

### Abstract & Clinical Context
Malignant melanoma is one of the deadliest forms of cutaneous cancer, responsible for the vast majority of skin cancer deaths despite accounting for only about 1% of skin cancers. Early detection is paramount: when detected and treated early, the 5-year survival rate exceeds 99%, but drops precipitously once metastatic spread occurs.

In clinical dermatology, practitioners utilize the **ABCD Rule** to assess suspicious lesions:
- **A (Asymmetry)**: Malignant lesions lack axial symmetry when bisected.
- **B (Border Irregularity)**: Melanomas exhibit notched, scalloped, or blurred borders rather than smooth circumscribed edges.
- **C (Color Variegation)**: Malignant melanomas display heterogeneous hues (shades of tan, brown, black, red, and blue-gray).
- **D (Diameter & Differential Structure)**: Dynamic enlargement, atypical textural grain, and irregular reticular networks.

In this study and implementation, we build an end-to-end computer vision and machine learning pipeline that extracts mathematically rigorous ABCD and textural descriptors using **OpenCV** and trains a **Support Vector Machine (SVM)** with a Radial Basis Function (RBF) kernel to classify dermoscopic images as **Benign** or **Malignant**.

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Ensure project modules are accessible
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.preprocessing import ImagePreprocessor
from src.feature_extraction import FeatureExtractor
from src.config import Config

print("All core scientific and ML modules imported successfully.")

### 1. Dataset Inspection and Class Distribution
We inspect the dataset metadata acquired from the International Skin Imaging Collaboration (ISIC) Archive, containing balanced ground-truth dermoscopic images.

In [ ]:
meta_path = ROOT_DIR / "data" / "dataset" / "dataset_metadata.csv"
df_meta = pd.read_csv(meta_path)

print("Dataset Summary:")
print(f"Total Images: {len(df_meta)}")
print("Class Distribution:")
print(df_meta['class'].value_counts())
df_meta.head()

### 2. OpenCV Preprocessing Demonstration
Dermoscopy images frequently contain visual artifacts such as patient hair, bubbles from immersion fluid, and peripheral lens vignetting. We utilize a multi-stage OpenCV preprocessing pipeline:
1. **Resize**: Standardized to 256x256.
2. **DullRazor Hair Removal**: Grayscale conversion, rectangular black-hat morphological filtering, binary thresholding, and Telea inpainting.
3. **Contrast Limited Adaptive Histogram Equalization (CLAHE)**: Performed on the Luminance (L) channel in CIE Lab color space to normalize lighting without altering pigment hue.
4. **Otsu Lesion Segmentation**: Isolates the central lesion contour and creates a binary mask.

In [ ]:
preprocessor = ImagePreprocessor(target_size=(256, 256))

# Sample benign and malignant paths
sample_ben = ROOT_DIR / "data" / "samples" / "sample_benign_nevus_1.jpg"
sample_mal = ROOT_DIR / "data" / "samples" / "sample_malignant_melanoma_1.jpg"

res_ben = preprocessor.process(str(sample_ben))
res_mal = preprocessor.process(str(sample_mal))

# Visualize stages for Benign vs Malignant
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
titles = ["Original Resized", "Hair Removed (DullRazor)", "Lesion Mask (Otsu)", "Contour Overlay"]

for idx, (res, label) in enumerate([(res_ben, "Benign Nevus"), (res_mal, "Malignant Melanoma")]):
    axes[idx, 0].imshow(cv2.cvtColor(res['original_resized'], cv2.COLOR_BGR2RGB))
    axes[idx, 0].set_title(f"{label} - {titles[0]}")
    axes[idx, 1].imshow(cv2.cvtColor(res['hair_removed'], cv2.COLOR_BGR2RGB))
    axes[idx, 1].set_title(f"{label} - {titles[1]}")
    axes[idx, 2].imshow(res['lesion_mask'], cmap='gray')
    axes[idx, 2].set_title(f"{label} - {titles[2]}")
    axes[idx, 3].imshow(cv2.cvtColor(res['contour_overlay'], cv2.COLOR_BGR2RGB))
    axes[idx, 3].set_title(f"{label} - {titles[3]}")
    for col in range(4):
        axes[idx, col].axis('off')

plt.tight_layout()
plt.show()

### 3. ABCD Feature Extraction Analysis
We extract 46 quantitative visual features per image across:
- **Color Moments (28 features)**: Mean, standard deviation, and skewness across BGR, HSV, and Lab color spaces, along with normalized color variegation.
- **Shape and Border (10 features)**: Contour area, perimeter, compactness ($4\pi A / P^2$), convexity solidity, aspect ratio, extent, and horizontal/vertical fold asymmetry indices.
- **Texture Properties (8 features)**: Fast Gray-Level Co-occurrence Matrix (GLCM) properties (contrast, dissimilarity, homogeneity, energy, correlation), Laplacian variance, and Sobel edge intensity statistics.

In [ ]:
extractor = FeatureExtractor()
summary_ben = extractor.extract_summary(res_ben)
summary_mal = extractor.extract_summary(res_mal)

comparison_df = pd.DataFrame([summary_ben, summary_mal], index=["Benign Nevus", "Malignant Melanoma"]).T
print("Comparison of Key Extracted ABCD Features:")
print(comparison_df)

**Analysis of Results:**
- **Asymmetry Index**: Malignant melanoma shows significantly higher asymmetry than the benign nevus.
- **Border Compactness & Irregularity**: Benign nevus exhibits high circular compactness, whereas melanoma shows jagged border irregularity.
- **Color Variegation**: Melanoma exhibits variegated multi-chromatic pigmentation resulting in substantially higher color variance.

### 4. Featurization Ordering & Model Training
Following Machine Learning best practices:
- We first split the dataset into stratified 80% train and 20% test sets.
- **StandardScaler** is fitted strictly on the training set to prevent data leakage.
- We evaluate a Support Vector Machine with an RBF kernel using CalibratedClassifierCV to obtain calibrated posterior probabilities.

In [ ]:
from scripts.train_model import extract_dataset_features

X, y, feature_names = extract_dataset_features()

# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calibrated SVM Classifier
base_svm = SVC(C=2.0, kernel="rbf", gamma="scale", class_weight="balanced", random_state=42)
calibrated_svm = CalibratedClassifierCV(estimator=base_svm, cv=5)

# 5-Fold Cross Validation on Train Split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(calibrated_svm, X_train_scaled, y_train, cv=cv, scoring="accuracy")
print(f"5-Fold Cross-Validation Accuracy on Train Set: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# Fit model on full training set
calibrated_svm.fit(X_train_scaled, y_train)

### 5. Evaluation on Held-Out Test Set
We evaluate model accuracy, precision, recall, and F1-score on the independent test split.

In [ ]:
y_pred = calibrated_svm.predict(X_test_scaled)
y_proba = calibrated_svm.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("=== Held-Out Test Set Evaluation ===")
print(f"Accuracy  : {acc * 100:.2f}%")
print(f"Precision : {prec * 100:.2f}%")
print(f"Recall    : {rec * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

# Plot Confusion Matrix
plt.figure(figsize=(6, 5), dpi=120)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Predicted Benign", "Predicted Malignant"],
            yticklabels=["Actual Benign", "Actual Malignant"],
            annot_kws={"size": 14, "weight": "bold"})
plt.title("Confusion Matrix - Held-Out Test Set", fontsize=13, weight="bold")
plt.tight_layout()
plt.show()

### 6. Clinical Decision Analysis & Conclusion

#### Clinical Decision Matrix Insights:
1. **Minimizing False Negatives (FN)**: In oncology screening, false negatives (failing to identify a malignant melanoma) are clinically dangerous. High sensitivity (Recall) is favored.
2. **Balancing False Positives (FP)**: False positives trigger biopsy referral; while inconvenient, they carry lower medical risk than untreated melanoma.
3. **Interpretability**: By coupling OpenCV ABCD visual criteria (asymmetry, circularity, variegation, texture) with the SVM model, clinicians and researchers gain transparent, auditable feature measurements rather than a pure black box.

#### Educational Disclaimer:
This system is intended as an educational screening and research tool. It is not an automated diagnostic device and does not replace evaluation by a board-certified dermatologist.